In [ ]:
import sys
import os
import pandas as pd
import time

In [49]:
measured = pd.read_csv('Manual_Foraging_Events_Observation.csv')

In [50]:
# load from final csv
predicted = pd.read_csv('CVPR_Evaluation_Results.csv')

predicted = predicted[['action', 'nest', 'frame_number',"video","timestamp","filename"]].copy()

In [51]:
predicted.dropna(inplace=True)

In [52]:
#predicted = 
predicted.reset_index(drop=True, inplace=True)

In [53]:
predicted['video'] = predicted['video'].apply(lambda x: x.replace('.mp4', ''))

In [55]:
predicted['timestamp'] = predicted['timestamp'].astype(str)
predicted['timestamp'] = predicted['timestamp'].apply(lambda x: x.split(' ')[1])

In [56]:
measured = measured[['video','action','nest','timestamp']].dropna()

In [57]:
from datetime import time
def getTimestamp1(txt):
    hr, mn, s = txt.split(':')
    #return timedelta(hours=int(hr), minutes=int(mn), seconds=int(s))
    return time(int(hr), int(mn), int(s))
measured['timestamp'] = measured['timestamp'].apply(getTimestamp1)

In [58]:
def getTimestamp2(txt):
    hr, mn, s = txt.split(':')
    #return timedelta(hours=int(hr), minutes=int(mn), seconds=int(s))
    return time(int(hr), int(mn), int(float(s)))
predicted['timestamp'] = predicted['timestamp'].apply(getTimestamp2)


In [60]:
measured['site'] = measured['video'].apply(lambda x: x.split('_')[0])
measured['hour'] = measured['timestamp'].apply(lambda x: x.hour)

predicted['site'] = predicted['video'].apply(lambda x: x.split('_')[0])
predicted['hour'] = predicted['timestamp'].apply(lambda x: x.hour)

In [61]:
# filter measured based on videos in predicted
videos = predicted.video.unique().tolist()
videos = [v.replace('.mp4', '') for v in videos]
measured_temp = measured[measured['video'].isin(videos)]

In [62]:
len(measured_temp)

300

In [63]:
measured.groupby('video').count()

,action,nest,timestamp,site,hour
video,,,,,
mendels_2024-04-30_09_00_00,10,10,10,10,10
mendels_2024-04-30_09_10_01,16,16,16,16,16
mendels_2024-04-30_09_20_00,9,9,9,9,9
mendels_2024-04-30_09_30_00,18,18,18,18,18
mendels_2024-04-30_09_40_01,5,5,5,5,5
mendels_2024-05-08_15_00_00,23,23,23,23,23
mendels_2024-05-08_15_30_00,26,26,26,26,26
mendels_2024-05-08_15_50_00,23,23,23,23,23
mendels_2024-05-23_12_00_00,27,27,27,27,27


In [64]:
predicted.groupby('video').count()

,action,nest,frame_number,timestamp,filename,site,hour
video,,,,,,,
mendels_2024-04-30_09_00_00,11,11,11,11,11,11,11
mendels_2024-04-30_09_10_01,14,14,14,14,14,14,14
mendels_2024-04-30_09_20_00,14,14,14,14,14,14,14
mendels_2024-04-30_09_30_00,19,19,19,19,19,19,19
mendels_2024-04-30_09_40_01,6,6,6,6,6,6,6
mendels_2024-05-08_15_00_00,34,34,34,34,34,34,34
mendels_2024-05-08_15_30_00,36,36,36,36,36,36,36
mendels_2024-05-08_15_50_00,50,50,50,50,50,50,50
mendels_2024-05-23_12_00_00,34,34,34,34,34,34,34


In [65]:
measured_temp.reset_index(drop=True, inplace=True)
predicted.reset_index(drop=True, inplace=True)

In [67]:
class Action:
    def __init__(self, action, timestamp, nest, video):
        self.action = action
        self.timestamp = timestamp
        self.nest = int(nest)
        self.video = video

    def getAction(self):
        return self.action
    
    def getTimestamp(self):
        return self.timestamp
    
    def getNest(self):
        return self.nest
    
    def getVideo(self):
        return self.video

def getActions(df):
    actions = []
    for i in range(len(df)):
        action = Action(df['action'][i], df['timestamp'][i], df['nest'][i], df['video'][i])
        actions.append(action)
    return actions

from datetime import datetime

def time_difference(time1, time2):
    # Convert the time strings to datetime objects
    date_today = datetime.today().date()
    datetime1 = datetime.combine(date_today, time1)
    datetime2 = datetime.combine(date_today, time2)

    # Calculate the difference
    time_difference = datetime1 - datetime2

    # Get the difference in seconds
    difference_in_seconds = time_difference.total_seconds()

    return abs(difference_in_seconds)

def isActionInActions(action, actions):
    for act in actions:

        if action.action == act.action and time_difference(action.timestamp, act.timestamp) < 3 and action.video == act.video and action.nest == act.nest:
            return True
        
    return False

In [68]:
measured_actions = getActions(measured_temp)

In [69]:
predicted_actions = getActions(predicted)


In [70]:
def calculateTruePositives(measured_actions, predicted_actions):
    tp = 0
    objs = []
    for action in predicted_actions:
        if isActionInActions(action, measured_actions):
            tp += 1
            objs.append(action)
    return tp, objs

tp, tp_obj = calculateTruePositives(measured_actions, predicted_actions)
print(tp)

251


In [71]:
tp_df = pd.DataFrame([obj.__dict__ for obj in tp_obj])
tp_df.groupby('action').count()

,timestamp,nest,video
action,,,
Entry,133,133,133
Exit,118,118,118


In [72]:
def calculateFalsePositives(measured_actions, predicted_actions):
    fp = 0
    fp_obj = []
    for action in predicted_actions:
        if not isActionInActions(action, measured_actions):
            fp += 1
            fp_obj.append(action)
    return fp, fp_obj

fp, fp_obj = calculateFalsePositives(measured_actions, predicted_actions)
print(fp)

130


In [73]:
fp_df = pd.DataFrame([obj.__dict__ for obj in fp_obj])
fp_df.groupby('action').count()

,timestamp,nest,video
action,,,
Entry,69,69,69
Exit,61,61,61


In [74]:
def calculateFalseNegatives(measured_actions, predicted_actions):
    fn = 0
    fn_obj = []
    for action in measured_actions:
        if not isActionInActions(action, predicted_actions):
            fn += 1
            fn_obj.append(action)
    return fn, fn_obj

fn, fn_obj = calculateFalseNegatives(measured_actions, predicted_actions)
print(fn)

50


In [75]:
fn_df = pd.DataFrame([obj.__dict__ for obj in fn_obj])
fn_df.groupby('action').count()

,timestamp,nest,video
action,,,
Entry,16,16,16
Exit,34,34,34


In [76]:
import numpy as np

In [77]:
# overall precision
np.mean(tp_df.groupby('video').size() / predicted.groupby('video').size()).tolist()

0.708643905478551

In [78]:
# precision per video
tp_df.groupby('video').size() / predicted.groupby('video').size()

video
mendels_2024-04-30_09_00_00    0.909091
mendels_2024-04-30_09_10_01    0.785714
mendels_2024-04-30_09_20_00    0.571429
mendels_2024-04-30_09_30_00    0.947368
mendels_2024-04-30_09_40_01    0.833333
mendels_2024-05-08_15_00_00    0.558824
mendels_2024-05-08_15_30_00    0.666667
mendels_2024-05-08_15_50_00    0.380000
mendels_2024-05-23_12_00_00    0.794118
mendels_2024-05-23_12_40_00    0.768293
mendels_2024-05-23_18_20_01    0.580247
dtype: float64

In [79]:
# recall per video
tp_df.groupby('video').size() / measured_temp.groupby('video').size()

video
mendels_2024-04-30_09_00_00    1.000000
mendels_2024-04-30_09_10_01    0.687500
mendels_2024-04-30_09_20_00    0.888889
mendels_2024-04-30_09_30_00    1.000000
mendels_2024-04-30_09_40_01    1.000000
mendels_2024-05-08_15_00_00    0.826087
mendels_2024-05-08_15_30_00    0.923077
mendels_2024-05-08_15_50_00    0.826087
mendels_2024-05-23_12_00_00    1.000000
mendels_2024-05-23_12_40_00    0.954545
mendels_2024-05-23_18_20_01    0.610390
dtype: float64

In [80]:
# overall recall
np.mean(tp_df.groupby('video').size() / measured_temp.groupby('video').size()).tolist()

0.8833249809040322